# Bay Area Households by County (ACS 1-Year Estimates)

Queries the Census Bureau ACS API for variable **B11001_001E** (total households)
across the 9 Bay Area counties for the last 10 years (2015–2024).

In [1]:
from census import Census
import pandas as pd

# Load Census API key
with open(r"M:\Data\Census\API\api-key.txt") as f:
    CENSUS_API_KEY = f.read().strip()

c = Census(CENSUS_API_KEY)

BAY_AREA_COUNTIES = {
    "001": "Alameda",
    "013": "Contra Costa",
    "041": "Marin",
    "055": "Napa",
    "075": "San Francisco",
    "081": "San Mateo",
    "085": "Santa Clara",
    "095": "Solano",
    "097": "Sonoma",
}

# 2020 ACS 1-year was not released due to COVID data quality issues
YEARS = [y for y in range(2015, 2025) if y != 2020]

VARIABLES = {
    "B11001_001E": "households",   # B11001: Household Type — total households
    "B01003_001E": "population",   # B01003: Total Population
}

In [2]:
records = []

county_list = ",".join(BAY_AREA_COUNTIES.keys())

for year in YEARS:
    rows = c.acs1.get(
        ("NAME", *VARIABLES.keys()),
        {"for": f"county:{county_list}", "in": "state:06"},
        year=year,
    )
    for row in rows:
        county_fips = row["county"]
        if county_fips in BAY_AREA_COUNTIES:
            record = {"year": year, "county": BAY_AREA_COUNTIES[county_fips]}
            for api_var, col_name in VARIABLES.items():
                record[col_name] = int(row[api_var])
            records.append(record)

df = pd.DataFrame(records).sort_values(["year", "county"]).reset_index(drop=True)

In [3]:
# Pivot to wide format: households and population
pivot = df.pivot(index="year", columns="county",
                 values=["households", "population"])

pivot.index.name = "Year"
pivot.sort_index(axis=1, inplace=True)

# Rename top-level metric labels to include Census table source
pivot.columns = pivot.columns.set_levels(
    ["households (B11001)", "population (B01003)"],
    level=0
)
pivot

households (B11001)                                            \
county             Alameda Contra Costa   Marin   Napa San Francisco   
Year                                                                   
2015                571828       391996  105887  49619        356916   
2016                572012       391288  106333  49561        358703   
2017                573589       392046  104219  47848        360323   
2018                575410       396133  104954  47315        362827   
2019                585632       399792  105298  48107        365851   
2021                589180       411560  103378  49979        350796   
2022                596614       415194  103285  51025        361912   
2023                608534       416172  101608  51167        372027   
2024                612675       417686  101974  49477        371841   

                                             population (B01003)               \
county San Mateo Santa Clara  Solano  Sonoma             Alameda Contra Costa   
Year                                                                            
2015      263280      633786  145502  190662             1638215      1126745   
2016      263445      635687  149172  187504             1647704      1135127   
2017      264185      633811  151127  188829             1663190      1147439   
2018      259654      645108  152291  187434             1666753      1150215   
2019      265003      643637  150393  190689             1671329      1153526   
2021      264135      650593  157617  190586             1648556      1161413   
2022      262122      656477  159294  194698             1628997      1156966   
2023      265124      665549  157766  192765             1622188      1155025   
2024      266249      672426  160415  193185             1649060      1172607   

                                                                            
county   Marin    Napa San Francisco San Mateo Santa Clara  Solano  Sonoma  
Year                                                                        
2015    261221  142456        864816    765135     1918044  436092  502146  
2016    260651  142166        870887    764797     1919402  440207  503070  
2017    260955  140973        884363    771410     1938153  445458  504217  
2018    259666  139417        883305    769545     1937570  446610  499942  
2019    258826  137744        881549    766573     1927852  447643  494336  
2021    260206  136207        815201    737888     1885508  451716  485887  
2022    256018  134300        808437    729181     1870945  448747  482650  
2023    254407  133216        808988    726353     1877592  449218  481812  
2024    256400  132727        827526    742893     1926325  455101  485375